# 📐 Etapa 1: Preparación, Limpieza y Enriquecimiento de Datos
**Proyecto:** Visual Analytics 2026 — StreamView Analytics  
**Asignatura:** ADY1104 Visualización de Datos (Duoc UC)  
**Responsable:** Persona 1 (Ingeniería y Preparación de Datos)  
**Fecha:** Septiembre 2026  

---

### 🎯 Objetivo del Notebook
Este notebook procesa, valida y enriquece los datasets originales en bruto (`netflix_movies_detailed_up_to_2025.csv` y `netflix_tv_shows_detailed_up_to_2025.csv`) para entregar a **Persona 2 (Visualización)** y **Persona 3 (Storytelling e Informe Ejecutivo)** archivos limpios, trazables y 100% confiables en `data/processed/`.

### 📋 Hoja de Ruta de Procesamiento:
1. **Carga y Diagnóstico Estructural:** Exploración de dimensiones y esquema original.
2. **Tratamiento de Inconsistencias de Tipo y Formato:** Depuración de columnas no utilizables (`duration`, `rating`) y conversión rigurosa de fechas (`date_added`).
3. **Deduplicación de Registros:** Eliminación de duplicados por `show_id` con aserciones inmediatas.
4. **Tratamiento de Nulos Textuales:** Imputación controlada con `"Sin dato"` para evitar pérdidas de cobertura en agrupaciones.
5. **Diagnóstico Formal de Outliers y Reglas Financieras (Películas):**
   - Diagnóstico exploratorio de distribuciones con percentiles.
   - Justificación fundamentada del umbral de **USD 1.000**.
   - Definición de columnas explícitas (`budget_valido`, `revenue_valido`, `tiene_datos_financieros`) sin imputación destructiva.
6. **Diagnóstico de Métricas de Calificación (`vote_count = 0`):** Evaluación del comportamiento de `vote_average` cuando no existen votos.
7. **Verificación Estructurada de Fechas de Estreno y Variables Derivadas:**
   - Verificación de integridad de `release_year`.
   - Generación de `decada` y cálculo del `ROI` (Retorno de Inversión) preservando formato `float64`.
8. **Documentación de Limitaciones Conocidas:** Insumos clave para la narrativa del informe ejecutivo.
9. **Validación Unitaria y Exportación:** Pruebas automáticas (`assert`) y exportación a `data/processed/`.

---
## 1. Carga de datos


In [1]:
import os
import pathlib
import numpy as np
import pandas as pd

# Definición de rutas relativas según la estructura estándar del repositorio
BASE_DIR = pathlib.Path().resolve().parent if pathlib.Path().resolve().name == 'notebooks' else pathlib.Path().resolve()
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DATA_DIR = BASE_DIR / 'data' / 'processed'

# Crear carpeta de destino si no existe
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Archivos de entrada
MOVIES_PATH = RAW_DATA_DIR / 'netflix_movies_detailed_up_to_2025.csv'
TV_PATH = RAW_DATA_DIR / 'netflix_tv_shows_detailed_up_to_2025.csv'

# Carga de datasets
movies_raw = pd.read_csv(MOVIES_PATH)
tv_raw = pd.read_csv(TV_PATH)

# Copias de trabajo
movies = movies_raw.copy()
tv = tv_raw.copy()

print(f"✅ Archivos cargados exitosamente:")
print(f" - Películas (Raw): {movies.shape[0]:,} filas x {movies.shape[1]} columnas")
print(f" - Series TV  (Raw): {tv.shape[0]:,} filas x {tv.shape[1]} columnas")

✅ Archivos cargados exitosamente:
 - Películas (Raw): 16,000 filas x 18 columnas
 - Series TV  (Raw): 16,000 filas x 16 columnas


---
## 2. Eliminar columnas sin valor analítico

## 🛠️ 1. Inconsistencias de Tipos, Estructura y Formatos

### Justificación de Modificaciones:
1. **Eliminación de `duration`:** En películas está completamente vacía (100% nula) y en series contiene un único valor estático que no aporta discriminación analítica.
2. **Eliminación de `rating`:** En ambos datasets es idéntica variable por variable a `vote_average`, representando una duplicidad de información redundante.
3. **Conversión de `date_added`:** Se convierte de texto a `datetime64[ns]`. Se implementa una validación posterior con `assert` para garantizar que la conversión con `errors='coerce'` no introdujo nulos adicionales por fallos de formato.

In [2]:
# 1. Eliminación de columnas con inconsistencias estructurales o redundancia
cols_to_drop = ['duration', 'rating']
movies.drop(columns=[c for c in cols_to_drop if c in movies.columns], inplace=True)
tv.drop(columns=[c for c in cols_to_drop if c in tv.columns], inplace=True)

print("✅ Columnas 'duration' y 'rating' eliminadas por inconsistencia y redundancia.")

# 2. Conversión segura de 'date_added' a datetime
for df, nombre in [(movies, 'movies'), (tv, 'tv')]:
    nulos_antes = df['date_added'].isna().sum()
    df['date_added'] = pd.to_datetime(df['date_added'].astype(str).str.strip(), errors='coerce')
    nulos_despues = df['date_added'].isna().sum()
    
    # Verificación estricta: No deben haberse creado nulos nuevos por fallos de parsing
    assert nulos_antes == nulos_despues, f"❌ Error: La conversión de fechas generó nulos inesperados en {nombre}"
    print(f"✅ Fechas convertidas a datetime en {nombre} (Nulos mantenidos: {nulos_despues})")

✅ Columnas 'duration' y 'rating' eliminadas por inconsistencia y redundancia.
✅ Fechas convertidas a datetime en movies (Nulos mantenidos: 0)
✅ Fechas convertidas a datetime en tv (Nulos mantenidos: 0)


---
## 3. Deduplicación

### 🧹 2. Identificación y Tratamiento de Duplicados

Se analiza la unicidad del identificador primario `show_id`. El proyecto establece que **cada registro representa un único contenido audiovisual**. Se remueven duplicados conservando la primera ocurrencia e inmediatamente se valida con una aserción unitaria.

In [3]:
# Diagnóstico de duplicados
dup_movies = movies['show_id'].duplicated().sum()
dup_tv = tv['show_id'].duplicated().sum()

print(f"Duplicados iniciales por show_id - Películas: {dup_movies} | Series: {dup_tv}")

# Deduplicación
movies.drop_duplicates(subset='show_id', keep='first', inplace=True)
tv.drop_duplicates(subset='show_id', keep='first', inplace=True)

# Assert inmediato post-transformación
assert movies['show_id'].duplicated().sum() == 0, "❌ Error: Quedaron duplicados en Películas"
assert tv['show_id'].duplicated().sum() == 0, "❌ Error: Quedaron duplicados en Series TV"

print(f"✅ Deduplicación completada. Registros actuales: Movies={len(movies):,}, TV={len(tv):,}")

Duplicados iniciales por show_id - Películas: 0 | Series: 9
✅ Deduplicación completada. Registros actuales: Movies=16,000, TV=15,991


---
## 4. Tratamiento de valores nulos (columnas de texto)

## Tratamiento de Valores Nulos (Variables Categóricas)

**Criterio Metodológico:**  
No es éticamente ni analíticamente correcto eliminar producciones completas debido a la ausencia de datos en campos como `director`, `cast` o `country`. Para mantener la representatividad en métricas globales de volumen, popularidad y año, se imputan estos nulos con el valor explícito `"Sin dato"`. Esto previene además la pérdida silenciosa de filas en agrupaciones (`groupby`).

In [4]:
text_cols = ['director', 'cast', 'country', 'genres', 'language']

for df, nombre in [(movies, 'Movies'), (tv, 'TV Shows')]:
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna('Sin dato')
            # Confirmación de imputación
            assert df[col].isna().sum() == 0, f"❌ Error: Quedaron nulos en {col} ({nombre})"

print("✅ Todos los nulos en columnas de texto fueron imputados con 'Sin dato'.")

✅ Todos los nulos en columnas de texto fueron imputados con 'Sin dato'.


---
## 5. Formato de fechas (Normalización)

`date_added` llega como texto libre; se convierte a tipo fecha real para poder ordenar, filtrar
por rango y graficar evolución temporal correctamente.

**Verificación importante:** `pd.to_datetime(..., errors='coerce')` convierte cualquier fecha con
formato irreconocible en `NaT` (nulo) *en silencio* — si no se revisa, se podrían perder fechas sin
darse cuenta. Se verifica explícitamente que esto no ocurrió.

In [5]:
nulos_antes_mov = movies['date_added'].isna().sum()
nulos_antes_tv = tv['date_added'].isna().sum()

for df in (movies, tv):
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

nulos_despues_mov = movies['date_added'].isna().sum()
nulos_despues_tv = tv['date_added'].isna().sum()

print(f"Movies -> nulos antes: {nulos_antes_mov}, nulos después de parsear: {nulos_despues_mov}")
print(f"TV     -> nulos antes: {nulos_antes_tv}, nulos después de parsear: {nulos_despues_tv}")

assert nulos_despues_mov == nulos_antes_mov, "¡Se perdieron fechas al convertir en movies!"
assert nulos_despues_tv == nulos_antes_tv, "¡Se perdieron fechas al convertir en tv!"
print("\nVerificado: ninguna fecha se perdió silenciosamente durante la conversión.")

Movies -> nulos antes: 0, nulos después de parsear: 0
TV     -> nulos antes: 0, nulos después de parsear: 0

Verificado: ninguna fecha se perdió silenciosamente durante la conversión.


---
## .6 Regla de Negocio: Filtrado Financiero

Diagnóstico de Outliers y Filtrado Financiero (Reglas de Negocio #3 y #6)

### A. Diagnóstico de Distribución y Outliers
La distribución de `budget` y `revenue` en la industria cinematográfica presenta una alta asimetría positiva (*right-skewed*), donde superproducciones coexisten con cine independiente. Los valores superiores extremos de cientos de millones de dólares **no se eliminan** porque corresponden a producciones comerciales reales verificables.

### B. Justificación del Umbral Mínimo de USD 1.000
Se observan registros con presupuestos e ingresos entre $1 y $999 USD. Si bien son valores mayores a cero, resultan **no plausibles** para producciones comerciales completas registradas en un catálogo internacional. Se establece un umbral conservador de **USD 1.000** para excluir datos no representativos o marcadores de posición (*placeholders*).

### C. No Imputación de Nulos Financieros
Los datos faltantes o inferiores al umbral **no se imputan con medias ni medianas**, ya que inventar datos financieros alteraría severamente el cálculo del ROI. Se estructuran banderas explícitas (`tiene_datos_financieros`, `budget_valido`, `revenue_valido`) para que Persona 2 filtre los datos sin eliminar las películas del catálogo general.

In [6]:
# 1. Diagnóstico exploratorio de distribuciones numéricas
print("--- RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS (Películas Raw) ---")
display(movies[['budget', 'revenue', 'popularity', 'vote_average', 'vote_count']].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

# 2. Aplicación de Reglas de Negocio Financieras (Solo Películas)
UMBRAL_MINIMO = 1000

# Creación de columnas financieras válidas
movies['budget_valido'] = np.where(movies['budget'] >= UMBRAL_MINIMO, movies['budget'], np.nan)
movies['revenue_valido'] = np.where(movies['revenue'] >= UMBRAL_MINIMO, movies['revenue'], np.nan)

# Bandera booleana de datos financieros completos
movies['tiene_datos_financieros'] = movies['budget_valido'].notna() & movies['revenue_valido'].notna()

# Garantizar que Series TV NO contengan datos financieros (Regla #3)
tv['budget_valido'] = np.nan
tv['revenue_valido'] = np.nan
tv['tiene_datos_financieros'] = False

# Verificación de consistencia
num_con_finanzas = movies['tiene_datos_financieros'].sum()
print(f"✅ Filtrado financiero aplicado.")
print(f" - Películas con información financiera sólida (>= $1,000 USD): {num_con_finanzas:,} ({num_con_finanzas/len(movies):.2%})")
print(f" - Series TV con datos financieros: {tv['tiene_datos_financieros'].sum()} (100% excluidas según Regla #3)")

--- RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS (Películas Raw) ---


,budget,revenue,popularity,vote_average,vote_count
count,1.600000e+04,1.600000e+04,16000.000000,16000.000000,16000.000000
mean,8.766792e+06,2.446308e+07,20.384728,5.956368,718.656125
std,2.912450e+07,1.116977e+08,68.610033,1.754741,2080.198316
min,0.000000e+00,0.000000e+00,3.860000,0.000000,0.000000
1%,0.000000e+00,0.000000e+00,4.531990,0.000000,0.000000
5%,0.000000e+00,0.000000e+00,5.556000,0.000000,0.000000
25%,0.000000e+00,0.000000e+00,7.840750,5.600000,53.000000
50%,0.000000e+00,0.000000e+00,10.913500,6.300000,138.000000
75%,2.200000e+06,1.654473e+06,17.336500,6.923000,422.000000
95%,5.000000e+07,1.176168e+08,55.091350,7.742100,3356.050000


✅ Filtrado financiero aplicado.
 - Películas con información financiera sólida (>= $1,000 USD): 3,523 (22.02%)
 - Series TV con datos financieros: 0 (100% excluidas según Regla #3)


---
## 7. Variables derivadas / KPI

Diagnóstico de Evaluaciones, Verificación de Años y Variables Derivadas

### A. Diagnóstico de Registros sin Evaluaciones (`vote_count = 0`)
Se examinan los registros sin votos para alertar a Persona 2 sobre su comportamiento al construir gráficos de Popularidad vs. Calificación.

### B. Verificación de `release_year`
Antes de generar la variable agrupada `decada`, se realiza una verificación previa de tipo de dato, nulos y rango de fechas plausibles (1900–2026).

### C. Cálculo de ROI
El Retorno sobre la Inversión ($ROI = \frac{\text{revenue\_valido}}{\text{budget\_valido}}$) se calcula **exclusivamente** sobre las películas con `tiene_datos_financieros == True`. Se inicializa con `np.nan` para asegurar que el tipo de dato final sea `float64`.

In [7]:
# 1. Diagnóstico de vote_count = 0
sin_votos_mov = (movies['vote_count'] == 0).sum()
sin_votos_tv = (tv['vote_count'] == 0).sum()
print(f"🔎 Películas sin evaluaciones (vote_count=0): {sin_votos_mov} ({sin_votos_mov/len(movies):.2%})")
print(f"🔎 Series sin evaluaciones (vote_count=0): {sin_votos_tv} ({sin_votos_tv/len(tv):.2%})")

# Verificar el valor de vote_average para registros sin votos
avg_vote_zero_mov = movies.loc[movies['vote_count'] == 0, 'vote_average'].unique()
print(f"   -> Calificación asignada a películas con 0 votos: {avg_vote_zero_mov}")

# 2. Verificación estricta de release_year
for df, nombre in [(movies, 'movies'), (tv, 'tv')]:
    assert df['release_year'].isna().sum() == 0, f"❌ Error: Nulos encontrados en release_year ({nombre})"
    assert pd.api.types.is_numeric_dtype(df['release_year']), f"❌ Error: release_year no es numérico ({nombre})"
    assert (df['release_year'] >= 1900).all() and (df['release_year'] <= 2026).all(), f"❌ Error: Años fuera de rango en {nombre}"

# 3. Cálculo de la variable 'decada'
movies['decada'] = (movies['release_year'] // 10) * 10
tv['decada'] = (tv['release_year'] // 10) * 10

# 4. Cálculo del ROI (Preservando float64)
movies['ROI'] = np.nan
mask_fin = movies['tiene_datos_financieros']
movies.loc[mask_fin, 'ROI'] = movies.loc[mask_fin, 'revenue_valido'] / movies.loc[mask_fin, 'budget_valido']

# Validación técnica de ROI
assert movies['ROI'].dtype == np.float64, "❌ Error: La columna ROI no es de tipo float64"
assert movies.loc[~mask_fin, 'ROI'].isna().all(), "❌ Error: Hay valores de ROI en filas sin datos financieros válidos"

print("✅ Variables derivadas 'decada' y 'ROI' calculadas correctamente.")

🔎 Películas sin evaluaciones (vote_count=0): 894 (5.59%)
🔎 Series sin evaluaciones (vote_count=0): 3666 (22.93%)
   -> Calificación asignada a películas con 0 votos: [0.]
✅ Variables derivadas 'decada' y 'ROI' calculadas correctamente.


---
## 8. Limitaciones conocidas (no se corrigen en esta etapa, quedan documentadas)

Las siguientes limitaciones deben ser consideradas por **Persona 2** al graficar y por **Persona 3** al redactar el Informe Ejecutivo y la sección de "Evaluación Crítica":

1. **Datos Financieros Parciales:** Solo un subconjunto de películas posee presupuesto e ingresos válidos ($\ge \$1,000$ USD). Ninguna serie posee datos financieros.
2. **Campos Multivalor:** Las columnas `genres` y `country` contienen valores separados por comas. No se desglosan en esta etapa para dar flexibilidad a Persona 2 según el tipo de gráfico requerido.
3. **Métricas de Popularidad:** `popularity` es un índice relativo de TMDB, no representa reproducciones reales en la plataforma.
4. **Contenidos sin Evaluaciones:** Los registros con `vote_count = 0` presentan `vote_average = 0.0`. Se recomienda a Persona 2 filtrar o anotar estos registros en el dispersograma de Popularidad vs Calificación para no distorsionar las líneas de tendencia.

In [8]:
# Evidencia de las limitaciones documentadas arriba (para que quede trazable en el notebook)
print("Ejemplo de country con múltiples valores:")
print(movies['country'].dropna().str.contains(',').sum(), "de", len(movies), "filas tienen más de un país")
print()
print("Películas con vote_count = 0:", (movies['vote_count'] == 0).sum())

Ejemplo de country con múltiples valores:
4058 de 16000 filas tienen más de un país

Películas con vote_count = 0: 894


---
## 9. Exportar datasets limpios

Se ejecutan pruebas finales unitarias para asegurar que los archivos exportados cumplen al 100% con los estándares exigidos antes de ser consumidos en la Etapa 2.

In [10]:
# BATERÍA DE PRUEBAS UNITARIAS DE SEGURIDAD
for df, nombre in [(movies, 'Películas'), (tv, 'Series')]:
    # A. Sin duplicados
    assert df['show_id'].duplicated().sum() == 0, f"Error final: Duplicados en {nombre}"
    # B. Sin nulos en texto
    for col in text_cols:
        assert df[col].isna().sum() == 0, f"Error final: Nulos en texto {col} ({nombre})"
    # C. Sin nulos en década
    assert df['decada'].isna().sum() == 0, f"Error final: Nulos en decada ({nombre})"

# Específicos de Películas
assert movies['ROI'].dtype == np.float64, "Error final: ROI no es float64"
assert (movies.loc[movies['tiene_datos_financieros'], 'ROI'] > 0).all(), "Error final: ROI inválido"

print("✅ Batería de pruebas unitarias superada con 0 errores.")

# EXPORTACIÓN A DATA/PROCESSED/
MOVIES_CLEAN_PATH = PROCESSED_DATA_DIR / 'netflix_movies_cleaned.csv'
TV_CLEAN_PATH = PROCESSED_DATA_DIR / 'netflix_tv_cleaned.csv'

movies.to_csv(MOVIES_CLEAN_PATH, index=False)
tv.to_csv(TV_CLEAN_PATH, index=False)

print("\n🎉 ¡PROCESO DE ETAPA 1 FINALIZADO CON ÉXITO!")
print(f"📁 Dataset Películas guardado en: {MOVIES_CLEAN_PATH}")
print(f"📁 Dataset Series TV guardado en:  {TV_CLEAN_PATH}")

✅ Batería de pruebas unitarias superada con 0 errores.

🎉 ¡PROCESO DE ETAPA 1 FINALIZADO CON ÉXITO!
📁 Dataset Películas guardado en: C:\Users\Admin\OneDrive\Escritorio\proyectoV\Proyecto-StreamView-Analytics\data\processed\netflix_movies_cleaned.csv
📁 Dataset Series TV guardado en:  C:\Users\Admin\OneDrive\Escritorio\proyectoV\Proyecto-StreamView-Analytics\data\processed\netflix_tv_cleaned.csv
